In [ ]:
import csv
import pandas as pd
import numpy as np

# 1. โหลดข้อมูล
df_final = pd.read_csv("/Users/chanokchonkarinrak/Documents/GitHub/AMR_Thesis/MDR/spatiotemperal/escherichia_coli/e_coli_all_regions.csv")

# ตัวแปรตาม (Beta 0-1)
df_final['y_beta'] = df_final['percentage'] / 100.0
df_final['y_beta'] = df_final['y_beta'].clip(lower=0.0001, upper=0.9999) 

# ---------------------------------------------------------
# แบบที่ 1: Sine/Cosine
# ---------------------------------------------------------
df_final['month_numeric'] = df_final['month'].astype(float)
df_final['sin_month'] = np.sin(2 * np.pi * df_final['month_numeric'] / 12)
df_final['cos_month'] = np.cos(2 * np.pi * df_final['month_numeric'] / 12)

# ---------------------------------------------------------
# แบบที่ 2: ฤดูแบบไทย
# ---------------------------------------------------------
def get_season(m):
    if m in [3, 4, 5]: 
        return 'summer'
    elif m in [6, 7, 8, 9]: 
        return 'rainy'
    else: 
        return 'winter' # 10, 11, 12, 1, 2

df_final['season'] = df_final['month'].apply(get_season)

# แปลงเป็นรหัส
season_map = {'summer': 1, 'rainy': 2, 'winter': 3}
df_final['season_id'] = df_final['season'].map(season_map)

# ID เชิงพื้นที่ (1-13)
df_final['region_id'] = df_final['region'].astype(int)

# ID เชิงเวลา (เรียงลำดับจากเดือนแรก=1 ไปจนถึงเดือนล่าสุด)
df_final['year_month'] = df_final['year'].astype(str) + df_final['month'].astype(str).str.zfill(2)
unique_time = sorted(df_final['year_month'].unique())
time_map = {tm: i+1 for i, tm in enumerate(unique_time)}
df_final['time_id'] = df_final['year_month'].map(time_map)

top_5_patterns = [
    "Cephems, Fluoroquinolones, Folate Pathway Antagonists, Penicillins",
    "Aminoglycosides, Cephems, Fluoroquinolones, Folate Pathway Antagonists, Penicillins",
    "Fluoroquinolones, Folate Pathway Antagonists, Penicillins",
    "Aminoglycosides, Cephems, Fluoroquinolones, Folate Pathway Antagonists, Penicillins, β-Lactam Combination Agents",
    "Aminoglycosides, Fluoroquinolones, Folate Pathway Antagonists, Penicillins"
]
top_5_patterns = [p.upper() for p in top_5_patterns] 
df_final['Resistant_Drug_Classes'] = df_final['Resistant_Drug_Classes'].astype(str).str.upper()

# กรองเอาเฉพาะข้อมูลที่มีรายชื่อยาตรงกับ top_5_patterns
df_final = df_final[df_final['Resistant_Drug_Classes'].isin(top_5_patterns)].copy()

# สร้าง Dictionary เพื่อใช้ map (Pattern -> 1, 2, 3, 4, 5)
mdr_map = {mdr: i+1 for i, mdr in enumerate(top_5_patterns)}
df_final['mdr_id_numeric'] = df_final['Resistant_Drug_Classes'].map(mdr_map)

print("--- ID กลุ่มยา ---")
for k, v in mdr_map.items():
    print(f"ID {v}: {k}")
print("---------------------------------")

print(f"เตรียม DataFrame (df_final) เสร็จสมบูรณ์! จำนวนแถวที่เหลือ: {len(df_final)}")

# บันทึก DataFrame ที่เตรียมไว้แล้ว
df_final.to_csv('eco_all_region_spatiotemporal_prepared.csv', index=False)